## Part 1: Indexing Pipeline:
### Step 1: Data Loading:

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# Load PDF
loader = PyPDFLoader("./data/data.pdf")

# Load all pages as Document objects
docs = loader.load()

print(f"Number of pages: {len(docs)}")
# print(docs[0].page_content)

### Step 2: Data Splitting / Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(docs)

# ----------------------------------------------------
# Assign a unique ID to every chunk
# ----------------------------------------------------

for chunk_id, chunk in enumerate(chunks):

    chunk.metadata["chunk_id"] = chunk_id

print("Number of chunks:", len(chunks))

print(chunks[0].metadata)

### Step 3: Data Conversion (Embeddings) and Vector Storage

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Load embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

# Create FAISS vector database
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

# Save the vector database
vector_store.save_local("./vector_store")

print(f"Indexed {len(chunks)} chunks.")
print("Vector database saved to ./vector_store")

## Part 2: Generation Pipeline
### Step 4: Retrieval:

In [ ]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 5}
)
def retrieve_context(query):
    """
    Retrieve the top-k chunks for a query.

    Returns
    -------
    docs : list[Document]
        Retrieved LangChain Document objects.

    context : str
        Concatenated text of the retrieved documents,
        suitable for prompting the LLM.
    """

    docs = retriever.invoke(query)

    context = "\n\n".join(
        [
            f"Document {i+1}:\n{doc.page_content}"
            for i, doc in enumerate(docs)
        ]
    )

    return docs, context

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# ----------------------------
# Load benchmark
# ----------------------------
queries = pd.read_csv("./data/final.csv")

k = 5

query_types = queries["query_type"].unique()

# ----------------------------
# Evaluate per query type
# ----------------------------
for qtype in query_types:

    subset = queries[queries["query_type"] == qtype]

    recall_scores = []
    precision_scores = []
    hit_scores = []
    mrr_scores = []
    ap_scores = []
    ndcg_scores = []

    print(f"\nEvaluating {qtype} queries...")

    for _, row in tqdm(subset.iterrows(), total=len(subset)):

        query = row["query"]

        relevant = set(
            int(x.strip())
            for x in str(row["relevant_chunk_ids"]).split(",")
        )

        docs = retriever.invoke(query)

        retrieved = [
            int(doc.metadata["chunk_id"])
            for doc in docs
        ]

        # ----------------------------
        # Recall@K
        # ----------------------------
        hits = len(set(retrieved) & relevant)

        recall_scores.append(
            hits / len(relevant)
        )

        # ----------------------------
        # Precision@K
        # ----------------------------
        precision_scores.append(
            hits / k
        )

        # ----------------------------
        # Hit Rate@K
        # ----------------------------
        hit_scores.append(
            int(hits > 0)
        )

        # ----------------------------
        # MRR
        # ----------------------------
        rr = 0

        for rank, chunk in enumerate(retrieved, start=1):

            if chunk in relevant:
                rr = 1 / rank
                break

        mrr_scores.append(rr)

        # ----------------------------
        # Average Precision
        # ----------------------------
        num_hits = 0
        precision_sum = 0

        for rank, chunk in enumerate(retrieved, start=1):

            if chunk in relevant:
                num_hits += 1
                precision_sum += num_hits / rank

        ap_scores.append(
            precision_sum / len(relevant)
        )

      
    print("=" * 60)
    print(f"Query Type   : {qtype}")
    print(f"Queries      : {len(subset)}")
    print(f"Recall@{k}    : {np.mean(recall_scores):.4f}")
    print(f"Precision@{k} : {np.mean(precision_scores):.4f}")
    print(f"Hit Rate@{k}  : {np.mean(hit_scores):.4f}")
    print(f"MRR          : {np.mean(mrr_scores):.4f}")
    print(f"MAP          : {np.mean(ap_scores):.4f}")
   

### Step 5 and Step 6: Augmentation and Generation:

In [ ]:
# ==========================================================
# Generate LLM Responses using Ollama
# ==========================================================

import time
import pandas as pd
from tqdm import tqdm
from openai import OpenAI

# ==========================================================
# Load Benchmark Dataset
# ==========================================================

queries = pd.read_csv("./data/final.csv")

# ==========================================================
# Ollama Client
# ==========================================================

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

responses = []

# ==========================================================
# Timing Statistics
# ==========================================================

start_time = time.perf_counter()

total_prompt_tokens = 0
total_completion_tokens = 0
total_tokens = 0

# ==========================================================
# Generate Responses
# ==========================================================

for _, row in tqdm(queries.iterrows(), total=len(queries)):

    query = row["query"]

    # ------------------------------------------------------
    # Retrieve Top-K Chunks
    # ------------------------------------------------------

    docs, context = retrieve_context(query)

    retrieved_chunk_ids = [
        str(doc.metadata["chunk_id"])
        for doc in docs
    ]

    # ------------------------------------------------------
    # Prompt
    # ------------------------------------------------------

    prompt = f"""
Answer the question using ONLY the retrieved documents.

If the answer cannot be found in the retrieved documents, reply exactly:

"I could not find an answer to your question in the retrieved documents."

Retrieved Documents:
{context}

Question:
{query}

Answer:
"""

    # ------------------------------------------------------
    # Generate Answer
    # ------------------------------------------------------

    response = client.chat.completions.create(
        model="gemma3:1b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.0
    )

    answer = response.choices[0].message.content.strip()

    # ------------------------------------------------------
    # Token Usage
    # ------------------------------------------------------

    if response.usage is not None:

        total_prompt_tokens += response.usage.prompt_tokens
        total_completion_tokens += response.usage.completion_tokens
        total_tokens += response.usage.total_tokens

    # ------------------------------------------------------
    # Save Everything Needed For Evaluation
    # ------------------------------------------------------

    responses.append({

        # Query Information
        "query": row["query"],
        "query_type": row["query_type"],

        # Ground Truth
        "ground_truth_answer": row["ground_truth_answer"],

        # Retrieval Ground Truth
        "relevant_chunk_ids": row["relevant_chunk_ids"],

        # Retrieval Output
        "retrieved_chunk_ids": ",".join(retrieved_chunk_ids),
        "retrieved_context": context,

        # Generation Output
        "llm_response": answer

    })

# ==========================================================
# Timing Statistics
# ==========================================================

end_time = time.perf_counter()

print(f"\nTotal inference time : {end_time - start_time:.2f} seconds")
print(f"Average time/query   : {(end_time - start_time)/len(queries):.2f} seconds")

if total_tokens > 0:

    print(f"Average prompt tokens     : {total_prompt_tokens/len(queries):.1f}")
    print(f"Average completion tokens : {total_completion_tokens/len(queries):.1f}")
    print(f"Average total tokens      : {total_tokens/len(queries):.1f}")

else:

    print("Token usage not returned by Ollama.")

# ==========================================================
# Save Results
# ==========================================================

baseline_rag = pd.DataFrame(responses)

baseline_rag.to_csv(
    "./data/baseline_rag.csv",
    index=False
)

print("\nSaved baseline_rag.csv")

In [9]:
# ==========================================================
# Evaluate Generation Metrics (Reference-based)
# ==========================================================

import pandas as pd
from tqdm import tqdm

from rouge_score import rouge_scorer
from bert_score import score as bertscore

# ==========================================================
# Load Results
# ==========================================================

baseline_rag = pd.read_csv("./data/baseline_rag.csv")

# ==========================================================
# ROUGE Scorer
# ==========================================================

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

predictions = []
references = []

generation_results = []

# ==========================================================
# Compute ROUGE
# ==========================================================

for _, row in tqdm(
    baseline_rag.iterrows(),
    total=len(baseline_rag)
):

    prediction = row["llm_response"]
    reference = row["ground_truth_answer"]

    rouge = scorer.score(reference, prediction)

    predictions.append(prediction)
    references.append(reference)

    generation_results.append({

        "query": row["query"],

        "query_type": row["query_type"],

        "ROUGE-1": rouge["rouge1"].fmeasure,

        "ROUGE-2": rouge["rouge2"].fmeasure,

        "ROUGE-L": rouge["rougeL"].fmeasure

    })

generation_df = pd.DataFrame(generation_results)

# ==========================================================
# Compute BERTScore
# ==========================================================

P, R, F1 = bertscore(

    predictions,

    references,

    lang="en",

    verbose=False

)

generation_df["BERTScore"] = F1.tolist()

# ==========================================================
# Save Per-query Metrics
# ==========================================================

generation_df.to_csv(
    "./data/generation_metrics.csv",
    index=False
)

print("Saved generation_metrics.csv")

# ==========================================================
# Overall Metrics
# ==========================================================

print("\nOverall Generation Metrics\n")

overall = (

    generation_df

    .drop(columns=["query", "query_type"])

    .mean()

)

print(overall)

# ==========================================================
# Metrics by Query Type
# ==========================================================

print("\nGeneration Metrics by Query Type\n")

metrics_by_type = (

    generation_df

    .groupby("query_type")

    .mean(numeric_only=True)

)

print(metrics_by_type)

100%|██████████| 20/20 [00:00<00:00, 2950.20it/s]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Saved generation_metrics.csv

Overall Generation Metrics

ROUGE-1      0.272549
ROUGE-2      0.175503
ROUGE-L      0.245936
BERTScore    0.866933
dtype: float64

Generation Metrics by Query Type

                ROUGE-1   ROUGE-2   ROUGE-L  BERTScore
query_type                                            
aggregation    0.423529  0.400000  0.411765   0.885903
comparison     0.178618  0.060000  0.140522   0.852861
factoid        0.270782  0.160369  0.270782   0.873666
summarization  0.217265  0.081645  0.160675   0.855302
